# Zava Model Router Fine-Tuning

This notebook walks through **fine-tuning the Azure Foundry Model Router** end-to-end using the bundled [`zava_enterprise`](../../Sample_Datasets/Model_Router_Fine_Tuning/zava_enterprise/) dataset (200 train / 100 test enterprise-operations prompts labelled for `gpt-5`, `gpt-5-mini`, and `gpt-5-nano` — chosen as a **representative subset** of the supported LLMs).

You will:

1. **Validate** the JSONL training and test files locally against the [Model Router schema](../../Sample_Datasets/Model_Router_Fine_Tuning/SCHEMA.md).
2. **Upload** the files to your Azure AI Foundry project.
3. **Submit** a fine-tuning job against the `model-router` base model.
4. **Monitor** the job to completion.
5. **Download** training metrics (`results.csv`).
6. **Deploy** the fine-tuned router via the Azure Management REST API.
7. **Test** the deployment with a sample prompt and see which underlying model was picked.

> 💡 **Heads-up — the 3-model subset is just an example.** Model Router fine-tuning supports labelling for **any subset of the [supported LLMs](https://learn.microsoft.com/en-us/azure/foundry/openai/concepts/model-router#supported-models)** (spanning GPT, Claude, Llama, DeepSeek, Grok, gpt-oss). The router you train will route between exactly the LLMs you label for.

> ⚠️ **Read [`README.md`](./README.md) first** for Model Router specific restrictions (the deployed router routes between the exact LLM set you labelled for; deploying requires the **Azure AI Owner** role).


## 1. Environment Setup

Install dependencies into the current kernel and load credentials from `.env`.

In [ ]:
# Safe to re-run — no-op if already installed
%pip install --quiet requests python-dotenv pandas tqdm

In [2]:
import json
import os
import time
from pathlib import Path
from urllib.parse import urlparse

import requests
from dotenv import load_dotenv

NOTEBOOK_DIR = Path.cwd()
# When opened from a different cwd, fall back to this file's directory if available
if not (NOTEBOOK_DIR / "README.md").exists():
    # Try the notebook's own directory if running via Jupyter
    try:
        NOTEBOOK_DIR = Path(__file__).parent  # type: ignore[name-defined]
    except NameError:
        pass

load_dotenv(NOTEBOOK_DIR / ".env")
print(f"Notebook dir: {NOTEBOOK_DIR}")

Notebook dir: <repo-root>\Demos\Zava_ModelRouter_FT


## 2. Configuration

All settings are read from environment variables — set them in a .env file at the demo root (copy from [.env.template](./.env.template)).

| Variable | Used by | Purpose |
|----------|---------|---------|
| AZURE_OPENAI_PROJECT_ENDPOINT | upload / submit / monitor / test | Azure AI Foundry **project** endpoint (e.g. https://<resource>.services.ai.azure.com/api/projects/<project>) |
| AZURE_OPENAI_API_KEY | upload / submit / monitor / test | API key for the project resource |
| AZURE_SUBSCRIPTION_ID, AZURE_RESOURCE_GROUP, AZURE_RESOURCE_NAME | deploy | Azure resource coordinates for the Management API |
| AZURE_FINETUNED_DEPLOYMENT_NAME | deploy / test | Name you choose for the new fine-tuned router deployment |


In [ ]:
# ── Data plane (upload / submit / monitor / test) ──
OPENAI_PROJECT_ENDPOINT = os.getenv("AZURE_OPENAI_PROJECT_ENDPOINT", "<your-project-endpoint>")
OPENAI_API_KEY          = os.getenv("AZURE_OPENAI_API_KEY",          "<your-azure-openai-api-key>")

# ── Control plane (deploy) ──
SUBSCRIPTION_ID = os.getenv("AZURE_SUBSCRIPTION_ID", "<your-subscription-id>")
RESOURCE_GROUP  = os.getenv("AZURE_RESOURCE_GROUP",  "<your-resource-group>")
RESOURCE_NAME   = os.getenv("AZURE_RESOURCE_NAME",   "<your-resource-name>")

FINETUNED_DEPLOYMENT_NAME = os.getenv("AZURE_FINETUNED_DEPLOYMENT_NAME", "zava-model-router-ft")

print("Project endpoint:     ", OPENAI_PROJECT_ENDPOINT)
print("Fine-tuned deployment:", FINETUNED_DEPLOYMENT_NAME)


## 3. Load and inspect the data format

Each line in the JSONL file is one prompt + per-model binary correctness `labels` + per-model `usage`. Both `labels` and `usage` are **required** on every row, and their key sets must match. See [`SCHEMA.md`](../../Sample_Datasets/Model_Router_Fine_Tuning/SCHEMA.md) for the full contract.

```json
{
  "messages": [{"role": "user", "content": "<your prompt>"}],
  "labels": {
    "gpt-5_2025-08-07":      1,
    "gpt-5-mini_2025-08-07": 1,
    "gpt-5-nano_2025-08-07": 0
  },
  "usage": {
    "gpt-5_2025-08-07":      {"prompt_tokens": 15, "completion_tokens": 1222},
    "gpt-5-mini_2025-08-07": {"prompt_tokens": 15, "completion_tokens": 702},
    "gpt-5-nano_2025-08-07": {"prompt_tokens": 15, "completion_tokens": 1349}
  }
}
```

`1` = the model answered correctly. The router learns to pick the **cheapest** model that's likely to be correct.

Below we point at the bundled Zava enterprise dataset, preview one record, and run a local structural check (no API calls) using the helper in `scripts/dataset_utils.py` — required fields, consistent model-key set across rows, binary label values, and matching `usage` keys.


In [ ]:
# Bundled dataset paths — swap DATA_DIR for your own folder if you bring your own data.
DATA_DIR = (NOTEBOOK_DIR / ".." / ".." / "Sample_Datasets" / "Model_Router_Fine_Tuning" / "zava_enterprise").resolve()
train_file = DATA_DIR / "zava_enterprise_train.jsonl"
val_file   = DATA_DIR / "zava_enterprise_test.jsonl"
assert train_file.exists(), f"Training file not found: {train_file}"
assert val_file.exists(),   f"Validation file not found: {val_file}"

# Preview the first training example
with open(train_file, "r", encoding="utf-8") as f:
    sample = json.loads(f.readline())

print("=== Sample Training Record ===")
print(json.dumps(sample, indent=2)[:2000])


In [ ]:
from scripts.dataset_utils import validate_jsonl

for label, path in (("train", train_file), ("test", val_file)):
    n, model_keys, errors = validate_jsonl(path)
    print(f"\n=== {label}: {path.name} ===")
    print(f"  rows:        {n}")
    print(f"  model keys:  {model_keys}")
    print(f"  errors:      {len(errors)}")
    for e in errors[:5]:
        print("   -", e)
    if errors:
        raise SystemExit("Fix validation errors before uploading.")


## 4. Upload Data

Upload the training (and optional validation) JSONL files to Azure Foundry via the REST API. We then wait until each file finishes server-side processing before submitting the fine-tuning job.

In [ ]:
from scripts.dataset_utils import upload_file, wait_for_file_processed

print("Uploading training file...")
train_resp = upload_file(train_file, OPENAI_PROJECT_ENDPOINT, OPENAI_API_KEY)
training_file_id = train_resp["id"]
print(f"  Training file ID: {training_file_id}")
wait_for_file_processed(training_file_id, OPENAI_PROJECT_ENDPOINT, OPENAI_API_KEY)

validation_file_id = None
if val_file and Path(val_file).exists():
    print("Uploading validation file...")
    val_resp = upload_file(val_file, OPENAI_PROJECT_ENDPOINT, OPENAI_API_KEY)
    validation_file_id = val_resp["id"]
    print(f"  Validation file ID: {validation_file_id}")
    wait_for_file_processed(validation_file_id, OPENAI_PROJECT_ENDPOINT, OPENAI_API_KEY)
else:
    print("No validation file — Azure will split training data 80:20 automatically.")


## 5. Submit the Fine-Tuning Job

Submit the job against the `model-router` base model.

> **Important:** the Model Router base requires the payload field `"trainingType": 1` (its global training type). Omitting it, or sending `0`/`"Standard"`, returns `400 invalidPayload — does not support fine-tuning with Standard TrainingType`. The helper below includes this field; don't remove it.

In [ ]:
BASE_MODEL      = "model-router"   # Model Router base — do NOT change this
SEED            = 105              # for reproducibility
JOB_API_VERSION = "v1"

def create_finetuning_job(training_file_id, validation_file_id=None, model=BASE_MODEL, seed=SEED):
    url = f"{OPENAI_PROJECT_ENDPOINT}/fine_tuning/jobs?api-version={JOB_API_VERSION}"
    headers = {"Content-Type": "application/json", "api-key": OPENAI_API_KEY}
    # NOTE: Model Router requires `trainingType: 1` (its global training type, not
    # the default Standard). Omitting the field, or sending `0`/`Standard`, returns
    # `400 invalidPayload — does not support fine-tuning with Standard TrainingType`.
    payload = {"model": model, "training_file": training_file_id, "trainingType": 1}
    if validation_file_id:
        payload["validation_file"] = validation_file_id
    if seed is not None:
        payload["seed"] = seed
    resp = requests.post(url, headers=headers, json=payload)
    if not resp.ok:
        print(f"Job submission failed ({resp.status_code}): {resp.text}")
        print(f"Request URL:    {url}")
        print(f"Request payload: {json.dumps(payload, indent=2)}")
    resp.raise_for_status()
    return resp.json()

print("Submitting fine-tuning job...")
job_response = create_finetuning_job(training_file_id, validation_file_id)
job_id = job_response["id"]
print(f"Fine-tuning job submitted!  Job ID: {job_id}")


## 6. Monitor the Job

Poll until the job reaches a terminal state (`succeeded`, `failed`, or `cancelled`). This can take minutes to hours depending on dataset size and queue depth. Track progress in [Azure AI Foundry](https://ai.azure.com/) → **Fine-tuning** while this cell runs.

In [ ]:
from IPython.display import HTML, display

POLL_INTERVAL_SECONDS = 300  # 5 minutes between status checks

def get_job_status(job_id):
    url = f"{OPENAI_PROJECT_ENDPOINT}/fine_tuning/jobs/{job_id}?api-version={JOB_API_VERSION}"
    resp = requests.get(url, headers={"api-key": OPENAI_API_KEY})
    resp.raise_for_status()
    return resp.json()

def poll_until_complete(job_id, interval=POLL_INTERVAL_SECONDS):
    terminal = {"succeeded", "failed", "cancelled"}
    while True:
        status = get_job_status(job_id)
        current = status.get("status", "unknown")
        print(f"Job {job_id} status: {current}")
        if current in terminal:
            return status
        time.sleep(interval)

def render_job_summary(status):
    s = status.get("status", "unknown")
    color = {"succeeded": "#1a7f37", "failed": "#cf222e", "cancelled": "#9a6700"}.get(s, "#57606a")
    rows = [
        ("Job ID",          status.get("id", "—")),
        ("Status",          f'<span style="color:{color};font-weight:600">{s.upper()}</span>'),
        ("Base model",      status.get("model", "—")),
        ("Fine-tuned model", status.get("fine_tuned_model") or "—"),
        ("Trained tokens",  f'{status.get("trained_tokens") or 0:,}'),
        ("Training file",   status.get("training_file", "—")),
        ("Validation file", status.get("validation_file") or "—"),
        ("Result files",    ", ".join(status.get("result_files") or []) or "—"),
    ]
    body = "".join(
        f'<tr><td style="padding:4px 12px 4px 0;color:#57606a;white-space:nowrap">{k}</td>'
        f'<td style="padding:4px 0;font-family:ui-monospace,Consolas,monospace">{v}</td></tr>'
        for k, v in rows
    )
    return HTML(
        '<div style="border:1px solid #d0d7de;border-radius:6px;padding:12px 16px;max-width:780px">'
        '<div style="font-weight:600;margin-bottom:8px">Fine-tuning job summary</div>'
        f'<table style="border-collapse:collapse;font-size:13px">{body}</table>'
        '</div>'
    )

final_status = poll_until_complete(job_id)
display(render_job_summary(final_status))


## 7. Download Result File

On success, Azure Foundry generates a `results.csv` with per-step training metrics. Download and preview it.

In [ ]:
from scripts.post_training import render_training_summary

if final_status.get("status") == "succeeded":
    result_files = final_status.get("result_files", [])
    if not result_files:
        print("No result files returned.")
    else:
        result_file_id = result_files[0]
        url = f"{OPENAI_PROJECT_ENDPOINT}/openai/v1/files/{result_file_id}/content"
        resp = requests.get(url, headers={"api-key": OPENAI_API_KEY})
        resp.raise_for_status()
        output_path = NOTEBOOK_DIR / "results.csv"
        with open(output_path, "wb") as f:
            f.write(resp.content)
        print(f"Saved {result_file_id} → {output_path}")

        render_training_summary(output_path)
else:
    print(f"Job did not succeed. Status: {final_status.get('status')}")


## 8. Deploy the Fine-Tuned Router

Deployment uses the **Azure Management REST API** (control plane), which needs an Azure AD token rather than the data-plane API key. The cell below uses `DefaultAzureCredential` from `azure-identity` to pick up your `az login` session (or VS Code / managed identity) automatically — no token variable needed.

> You need the **Azure AI Owner** role on the resource to create deployments. If you see `CredentialUnavailableError`, run `az login` (and `az account set --subscription <id>` if needed).

> ♻️ **Re-runs are safe.** If a deployment with `FINETUNED_DEPLOYMENT_NAME` already exists, the cell deletes it first and then creates a fresh one. This is necessary because Azure rejects in-place model updates for fine-tuned deployments (`ModelUpgradeNotSupported`).

> ⚠️ **The deployed router routes between the exact LLMs you labelled for.** In this notebook that's `gpt-5`, `gpt-5-mini`, and `gpt-5-nano` — chosen as a representative subset. To target a different set, build a JSONL labelled for those LLMs (see the canonical [supported LLMs](https://learn.microsoft.com/en-us/azure/foundry/openai/concepts/model-router#supported-models) list) and re-run the notebook. You can deploy the resulting router in any [Model Router mode](https://learn.microsoft.com/azure/ai-services/openai/how-to/model-router) supported by your subscription.


In [ ]:
from azure.identity import DefaultAzureCredential
from scripts.post_training import deploy_finetuned_router

fine_tuned_model = final_status.get("fine_tuned_model")
print(f"Fine-tuned model: {fine_tuned_model}")
assert fine_tuned_model, "No fine-tuned model name on the job — ensure the job succeeded."

# DefaultAzureCredential picks up your `az login` session (or VS Code / managed identity)
# automatically. The signed-in identity must have the "Azure AI Owner" role on the resource.
aad_token = DefaultAzureCredential().get_token("https://management.azure.com/.default").token

deploy_finetuned_router(
    token=aad_token,
    fine_tuned_model=fine_tuned_model,
    deployment_name=FINETUNED_DEPLOYMENT_NAME,
    subscription_id=SUBSCRIPTION_ID,
    resource_group=RESOURCE_GROUP,
    resource_name=RESOURCE_NAME,
    project_endpoint=OPENAI_PROJECT_ENDPOINT,
)

## 9. Test the Deployed Router

Send a single prompt to the fine-tuned deployment and print (a) which underlying model the router picked and (b) a short preview of the response. The `model` field in the response is the routing decision — that's what fine-tuning was designed to change.

In [32]:
INFERENCE_API_VERSION = "2025-04-01-preview"

# Resolve the inference endpoint (strip the /api/projects/... path from the project endpoint)
_parsed = urlparse(OPENAI_PROJECT_ENDPOINT)
INFERENCE_ENDPOINT = f"{_parsed.scheme}://{_parsed.netloc}"

test_messages = [
    {"role": "user", "content": "Write a brief, friendly Slack announcement (3-4 sentences) letting the team know that the weekly engineering sync has moved from Monday 10am to Tuesday 2pm starting next week."}
]

print("=== Test Prompt ===")
print(test_messages[0]["content"])
print()

url = f"{INFERENCE_ENDPOINT}/openai/deployments/{FINETUNED_DEPLOYMENT_NAME}/chat/completions?api-version={INFERENCE_API_VERSION}"
headers = {"Content-Type": "application/json", "api-key": OPENAI_API_KEY}
resp = requests.post(url, headers=headers, json={"messages": test_messages})
resp.raise_for_status()
result = resp.json()

picked_model = result.get("model")
reply = result["choices"][0]["message"]["content"]
preview = reply[:300] + ("…" if len(reply) > 300 else "")

print(f"Fine-tuned router ({FINETUNED_DEPLOYMENT_NAME})  →  picked `{picked_model}`")
print()
print("=== Response (first 300 chars) ===")
print(preview)


=== Test Prompt ===
Write a brief, friendly Slack announcement (3-4 sentences) letting the team know that the weekly engineering sync has moved from Monday 10am to Tuesday 2pm starting next week.

Fine-tuned router (zava-model-router-ft)  →  picked `gpt-5-mini-2025-08-07`

=== Response (first 300 chars) ===
Quick update: starting next week our weekly engineering sync is moving from Monday at 10:00 AM to Tuesday at 2:00 PM. I’ve updated the calendar invite — please accept the new time if you haven’t already. If that creates a conflict, ping me and we’ll work it out.


## Next Steps

- **Bring your own prompts** — replace the Zava dataset with your own enterprise prompts. Label them against the LLM subset you want to route between — see the canonical [supported LLMs](https://learn.microsoft.com/en-us/azure/foundry/openai/concepts/model-router#supported-models) list on Microsoft Learn. The GPT-5 trio here is just a representative subset.
- **Evaluate** — compare your fine-tuned router against the stock `model-router` deployment on a held-out test set to measure accuracy + cost improvement. The [`microsoft-foundry/Model-Router-Auto-Evaluation`](https://github.com/microsoft-foundry/Model-Router-Auto-Evaluation) repo is a turnkey harness dedicated to Model Router evaluation — point it at both deployments and it produces side-by-side accuracy/cost metrics.

See the [Microsoft Foundry Model Router docs](https://learn.microsoft.com/azure/ai-services/openai/how-to/model-router) for runtime concepts, pricing, and the full list of supported models.
